# MIA — Kendi KKD Modelimizi Eğitme (v1 → v2)

**Amaç:** Topluluk ağırlıklarından kurtulmak ve **gözlük + eldiven** gibi bugün
kilitli olan sınıfları açmak.

**Nasıl kullanılır:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU** seç.
Sonra `Çalışma zamanı → Tümünü çalıştır`. Toplam ~2-4 saat (ücretsiz T4).

**Çıktı:** `mia-ppe-vX.onnx` — indirip `apps/desktop/models/mia-ppe-yolov8s.onnx`
üzerine yazarsın; uygulama başka değişiklik istemez.

> DÜRÜSTLÜK KURALI: Aday model, eval kapısında mevcut modeli **geçmeden**
> yayınlanmaz. Bu defter kapıyı otomatik uygular.

## 0) Ortam kontrolü — GPU var mı?

In [ ]:
!nvidia-smi
import torch; print("CUDA:", torch.cuda.is_available())
# GPU YOKSA: Çalışma zamanı → Çalışma zamanı türünü değiştir → Donanım hızlandırıcı: T4 GPU

## 1) Kurulum

In [ ]:
%pip -q install ultralytics==8.3.* onnx onnxruntime roboflow
import ultralytics; ultralytics.checks()

## 2) Veri seti seçimi — v1 mi v2 mi?

| | v1 (mevcut) | **v2 (önerilen)** |
|---|---|---|
| Veri seti | Construction Site Safety | **PPE Combined Model** |
| Görüntü | ~2.8k | **44k** |
| Sınıf | 10 | **14** |
| Gözlük | ✗ | **✓ Goggles / NO-Goggles** |
| Eldiven | ✗ | **✓ Gloves / NO-Gloves** |
| Ekstra | machinery, vehicle | **Fall-Detected, Ladder** |

**v2'yi seç** — gözlük ve eldiven kilidini açan tek yol bu. Ayrıca düşme (Fall-Detected)
ve merdiven tespiti bonus olarak gelir.

> Roboflow API anahtarı: roboflow.com → hesap aç (ücretsiz) → Settings → API Key.

In [ ]:
ROBOFLOW_API_KEY = ""   # ← roboflow.com → Settings → API Key
VERSION = "v2"          # "v1" (10 sınıf) veya "v2" (14 sınıf — gözlük+eldiven açılır)

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

if VERSION == "v2":
    # 44k görüntü · 14 sınıf · gözlük + eldiven + düşme + merdiven DAHİL
    ds = (rf.workspace("roboflow-universe-projects")
            .project("personal-protective-equipment-combined-model")
            .version(8).download("yolov8", location="/content/ppe"))
else:
    ds = (rf.workspace("roboflow-universe-projects")
            .project("construction-site-safety")
            .version(30).download("yolov8", location="/content/ppe"))
print("Veri:", ds.location)

# Veri setinin GERÇEK sınıf sırasını oku — varsayma, doğrula.
import yaml, pathlib
dy = yaml.safe_load(open(pathlib.Path(ds.location)/"data.yaml"))
DATASET_CLASSES = dy["names"]
print("Sınıflar:", DATASET_CLASSES)

### 3) MIA saha verisi (isteğe bağlı ama en değerli)

Masaüstü uygulama → Ayarlar → **Saha Veri Toplama Modu** (KVKK onayı şart).
Toplanan kareler `~/Library/Application Support/mia-desktop/mia-dataset/`.
Drive'a yükle, CVAT/Label Studio'da etiketleri **düzelt** (model ön-etiketi ham veridir),
sonra aşağıda `USE_DRIVE = True` yap.

TR şantiyesi verisi (beyaz/sarı baret kültürü, turuncu yelek, iskele arka planı)
kamu veri setlerinin yakalayamadığı farkı yaratır.

In [ ]:
USE_DRIVE = False
MIA_DATA_PATH = "/content/drive/MyDrive/mia-dataset"
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    import os; print("MIA saha verisi:", os.path.exists(MIA_DATA_PATH))

## 4) Veri hazırlığı

Veri setinin kendi sınıf sırası korunur (yeniden eşleme yapılmaz — hata kaynağı).
MIA saha verisi eklenirse etiketlerinin AYNI sınıf sırasında olması gerekir.

In [ ]:
import os, glob, random, shutil, yaml
from pathlib import Path

CLASSES = DATASET_CLASSES          # veri setinden okundu — elle yazma
OUT = Path("/content/mia_data")

def collect(root):
    pairs = []
    for img in Path(root).rglob("*.jpg"):
        if "images" not in img.parts: continue
        lbl = Path(str(img).replace("/images/","/labels/")).with_suffix(".txt")
        if lbl.exists(): pairs.append((img, lbl))
    return pairs

pairs = collect("/content/ppe")
print("Kamu verisi:", len(pairs))
if USE_DRIVE and os.path.exists(MIA_DATA_PATH):
    mia = collect(MIA_DATA_PATH); pairs += mia
    print(" + MIA saha verisi:", len(mia))

def valid(lbl):
    try: lines = Path(lbl).read_text().strip().splitlines()
    except OSError: return False
    if not lines: return False
    for ln in lines:
        p = ln.split()
        if len(p) != 5: return False
        try: c = int(p[0]); vals = [float(x) for x in p[1:]]
        except ValueError: return False
        if not (0 <= c < len(CLASSES)) or any(v < 0 or v > 1.5 for v in vals): return False
    return True

pairs = [(i,l) for i,l in pairs if valid(l)]
print("Geçerli toplam:", len(pairs))
assert len(pairs) > 500, "Veri çok az — indirme adımını kontrol et"

random.Random(42).shuffle(pairs)
n_val = int(len(pairs)*0.15)
for sp, items in {"val": pairs[:n_val], "train": pairs[n_val:]}.items():
    (OUT/sp/"images").mkdir(parents=True, exist_ok=True)
    (OUT/sp/"labels").mkdir(parents=True, exist_ok=True)
    for i,(img,lbl) in enumerate(items):
        stem = f"{sp}_{i:06d}"
        shutil.copyfile(img, OUT/sp/"images"/(stem+".jpg"))
        shutil.copyfile(lbl, OUT/sp/"labels"/(stem+".txt"))
    print(sp, len(items))

(OUT/"data.yaml").write_text(yaml.dump({
    "path": str(OUT), "train": "train/images", "val": "val/images",
    "nc": len(CLASSES), "names": CLASSES}, allow_unicode=True))
print("✔ data.yaml hazır ·", len(CLASSES), "sınıf")

## 5) Eğitim

~2-4 saat (T4). v2 (44k görüntü) v1'den uzun sürer — 60 epoch ile başlayıp
sonuç yeterliyse durabilirsin. Augmentasyon TR şantiye koşullarına göre ayarlı.

In [ ]:
# EĞİTİM — Colab kopabilir; ağırlıklar Drive'a yazılırsa kayıp OLMAZ.
SAVE_TO_DRIVE = True     # önerilen: True (bağlantı koparsa resume edebilirsin)
EPOCHS = 60              # v2/44k için 60 iyi başlangıç; sonuç yeterliyse artırma

import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    if not os.path.exists("/content/drive"): drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/mia-training"
else:
    PROJECT = "/content/runs"
os.makedirs(PROJECT, exist_ok=True)
print("Ağırlıklar buraya yazılacak:", PROJECT)

from ultralytics import YOLO
RUN_NAME = f"mia-ppe-{VERSION}"
ckpt = f"{PROJECT}/{RUN_NAME}/weights/last.pt"

if os.path.exists(ckpt):
    print("↻ Yarım kalan eğitim bulundu — kaldığı yerden devam ediyor")
    model = YOLO(ckpt)
    results = model.train(resume=True)
else:
    model = YOLO("yolov8s.pt")          # COCO ön-eğitimli taban
    results = model.train(
        data=str(OUT/"data.yaml"), epochs=EPOCHS, imgsz=640, batch=16,
        name=RUN_NAME, project=PROJECT, patience=20, seed=42,
        hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,     # toz / ışık varyasyonu
        degrees=8, scale=0.5, fliplr=0.5,      # açı + uzak kamera + ayna
        mosaic=1.0, close_mosaic=15,
    )
best = f"{PROJECT}/{RUN_NAME}/weights/best.pt"
print("✔ Eğitim bitti:", best)

## 6) EVAL KAPISI — kritik sınıflarda precision ≥ 0.80 olmadan YAYINLAMA YOK

In [ ]:
# Ultralytics'in kendi doğrulaması: sınıf bazlı precision/recall/mAP
m = YOLO(best)
metrics = m.val(data=str(OUT/"data.yaml"), split="val")

import numpy as np
names = m.names
p, r = metrics.box.p, metrics.box.r
f1 = 2*p*r/np.maximum(1e-9, p+r)
print(f"\n{'Sınıf':<20}{'P':>7}{'R':>7}{'F1':>7}")
for i, n in names.items():
    print(f"{n:<20}{p[i]:>7.2f}{r[i]:>7.2f}{f1[i]:>7.2f}")

# KAPI KURALI: kritik ihlal sınıflarında precision >= 0.80
CRITICAL = ["NO-Hardhat", "NO-Safety Vest"]
gate_ok = True
for n in CRITICAL:
    idx = [i for i,v in names.items() if v == n]
    if not idx:
        print(f"⚠ {n} bu modelde yok"); continue
    i = idx[0]
    if p[i] < 0.80:
        print(f"✘ KAPI: {n} precision {p[i]:.2f} < 0.80"); gate_ok = False
    else:
        print(f"✔ {n} precision {p[i]:.2f}")
print("\n" + ("✔ EVAL KAPISI GEÇİLDİ — yayınlanabilir" if gate_ok
      else "✘ KAPI GEÇİLEMEDİ — veri ekle / tekrar eğit. KISAYOL YOK."))

## 8) ONNX'e çevir ve indir

Kapı geçtiyse çalıştır. İnen dosyayı `apps/desktop/models/mia-ppe-yolov8s.onnx`
üzerine kopyala → `npm start` ile duman testi → `npm run release:mac`.

## 7) ONNX'e çevir, model.json üret ve indir

**ÖNEMLİ:** Uygulama sınıf listesini `models/model.json`'dan okur. İki dosyayı da indir:
`mia-ppe-v2.onnx` + `model.json`. İkisini birden `apps/desktop/models/` içine koy.

In [ ]:
onnx_path = YOLO(best).export(format="onnx", imgsz=640, simplify=True)
print("ONNX:", onnx_path)

import json, hashlib, shutil
shutil.copyfile(onnx_path, f"/content/mia-ppe-{VERSION}.onnx")
sha = hashlib.sha256(open(f"/content/mia-ppe-{VERSION}.onnx","rb").read()).hexdigest()

# Uygulamanın okuyacağı model tanımı — sınıf listesi BURADAN gelir.
model_json = {
    "_comment": "MIA model tanımı — uygulama KKD kilitlerini bu sınıf listesine göre açar.",
    "name": f"mia-ppe-{VERSION}",
    "version": VERSION,
    "file": f"mia-ppe-{VERSION}.onnx",
    "input_size": 640,
    "classes": CLASSES,
    "trained_on": f"Roboflow PPE ({len(pairs)} görüntü) + MIA saha verisi" if USE_DRIVE else f"Roboflow PPE ({len(pairs)} görüntü)",
    "field_validated": False,
    "sha256": sha,
    "notes": "Saha doğrulaması yapılmadı. Kilitler model sınıflarına göre otomatik açılır."
}
with open("/content/model.json","w",encoding="utf-8") as f:
    json.dump(model_json, f, ensure_ascii=False, indent=2)
print(json.dumps(model_json, ensure_ascii=False, indent=2))

from google.colab import files
files.download(f"/content/mia-ppe-{VERSION}.onnx")
files.download("/content/model.json")

## 8) Uygulamaya kurma (2 dakika)

1. İnen iki dosyayı kopyala:
   - `mia-ppe-v2.onnx` → `apps/desktop/models/`
   - `model.json` → `apps/desktop/models/model.json` (üzerine yaz)
2. Test et:
   ```bash
   cd apps/desktop && MIA_DEBUG=1 npm start
   ```
3. **Canlı İzleme** → çip çubuğuna bak: **Koruyucu Gözlük ve Eldiven artık kilitli DEĞİL** —
   tıklanabilir ve β (deneysel) işaretli olmalı.

Kod değişikliği GEREKMEZ. Uygulama model.json'daki sınıf listesini okur ve
`ppe-registry.bind()` ile kilitleri otomatik açar. Model geri alınırsa kilitler
kendiliğinden kapanır — olmayan yeteneği vaat etme riski yapısal olarak yok.

### Saha doğrulaması (deneysel → destekleniyor terfisi)
Yeni sınıflar `experimental` (β) olarak gelir. `supported` olması için:
pilot sahada ≥2 hafta gözlem + yanlış pozitif oranı ölçümü. Terfi için
`ppe-registry.js` içinde `status: "requires_training"` → `"supported"` yap
(web `js/ppe-registry.js` ve worker `.py` ile senkron).